# Smart Glove ASL Gesture Recognition with Conv1D + Bi-LSTM
# 智能手套ASL手势识别 - 时间序列深度学习方案

## 项目概述

本Notebook提供完整的**Conv1D + Bi-LSTM**模型训练方案，用于时间序列手势识别。支持从数据预处理到ESP32-S3边缘部署的全流程。

### 技术栈
- **模型架构**: Conv1D + Bi-LSTM (适合时间序列特征学习)
- **输入数据**: 11维特征 × 100帧时间序列 (5弯曲传感器 + 3加速度 + 3陀螺仪)
- **采样率**: 50Hz (20ms间隔)
- **部署目标**: ESP32-S3 with TensorFlow Lite Micro

### 数学基础

#### 1. 时间序列建模
输入张量形状: $X \in \mathbb{R}^{T \times F}$
- $T = 100$: 时间步长 (2秒窗口 @ 50Hz)
- $F = 11$: 特征维度

#### 2. Conv1D 特征提取
卷积操作: $y_t = \sum_{k=0}^{K-1} w_k \cdot x_{t+k} + b$
- 局部时序特征提取
- 平移不变性

#### 3. Bi-LSTM 长期依赖建模
前向LSTM: $\overrightarrow{h_t} = \text{LSTM}(x_t, \overrightarrow{h_{t-1}})$

后向LSTM: $\overleftarrow{h_t} = \text{LSTM}(x_t, \overleftarrow{h_{t+1}})$

拼接输出: $h_t = [\overrightarrow{h_t}; \overleftarrow{h_t}]$

#### 4. 分类输出
Softmax概率: $P(y=i|x) = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}$

## 第一步：环境配置与依赖安装

In [ ]:
# 安装必要的依赖库
# 取消注释以下行以安装依赖

# !pip install tensorflow numpy pandas matplotlib seaborn scikit-learn scipy

# 验证安装
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from scipy import signal
import os
import glob
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow版本: {tf.__version__}")
print(f"NumPy版本: {np.__version__}")
print(f"Pandas版本: {pd.__version__}")
print(f"GPU可用: {tf.config.list_physical_devices('GPU')}")

## 第二步：数据加载与探索

In [ ]:
# 配置参数
CONFIG = {
    'data_path': 'modified dataset/alphabet/',  # 数据目录
    'sequence_length': 100,  # 时间序列长度 (2秒 @ 50Hz)
    'sampling_rate': 50,     # 采样率 Hz
    'stride': 10,           # 滑动窗口步长
    'num_features': 11,     # 特征维度 (5 flex + 3 accel + 3 gyro)
    'test_size': 0.2,       # 测试集比例
    'random_state': 42
}

# 手势类别列表
# 根据项目实际数据调整
GESTURE_CLASSES = [
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 
    'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't',
    'u', 'v', 'w', 'x', 'y', 'z',
    'bad', 'deaf', 'fine', 'good', 'goodbye', 'hello', 
    'hungry', 'me', 'no', 'please', 'sorry', 'thankyou', 
    'yes', 'you'
]

print(f"配置参数:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\n总类别数: {len(GESTURE_CLASSES)}")

In [ ]:
# 加载单个CSV文件示例
def load_single_file(filepath):
    """
    加载单个手势数据文件
    
    参数:
        filepath: CSV文件路径
    
    返回:
        DataFrame: 包含传感器数据
    """
    df = pd.read_csv(filepath)
    
    # 选择核心特征列
    feature_cols = [
        'flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5',  # 5弯曲传感器
        'ACCx', 'ACCy', 'ACCz',                             # 3轴加速度
        'GYRx', 'GYRy', 'GYRz'                              # 3轴陀螺仪
    ]
    
    # 检查并过滤负值的弯曲传感器数据
    flex_cols = ['flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5']
    df[flex_cols] = df[flex_cols].clip(lower=0)
    
    # 删除包含NaN的行
    df = df.dropna(subset=feature_cols)
    
    return df

# 测试加载一个文件
test_file = os.path.join(CONFIG['data_path'], 'a_merged.csv_exported.csv')
if os.path.exists(test_file):
    test_df = load_single_file(test_file)
    print(f"成功加载文件: {test_file}")
    print(f"数据形状: {test_df.shape}")
    print(f"\n前几行数据:")
    print(test_df[['Alphabet', 'flex_1', 'flex_2', 'GYRx', 'ACCx']].head())
else:
    print(f"文件不存在: {test_file}")
    print(f"请确保数据路径正确: {os.path.abspath(CONFIG['data_path'])}")

In [ ]:
# 批量加载所有数据
def load_all_data(data_path, gesture_classes):
    """
    批量加载所有手势数据
    
    参数:
        data_path: 数据目录路径
        gesture_classes: 手势类别列表
    
    返回:
        tuple: (数据列表, 标签列表, 类别统计)
    """
    all_data = []
    all_labels = []
    class_counts = {}
    
    for gesture in gesture_classes:
        # 构建文件路径
        filename = f"{gesture}_merged.csv_exported.csv"
        filepath = os.path.join(data_path, filename)
        
        if os.path.exists(filepath):
            try:
                df = load_single_file(filepath)
                
                if len(df) > 0:
                    all_data.append(df)
                    all_labels.extend([gesture] * len(df))
                    class_counts[gesture] = len(df)
                    print(f"✓ {gesture}: {len(df)} 样本")
                else:
                    print(f"⚠ {gesture}: 数据为空")
            except Exception as e:
                print(f"✗ {gesture}: 加载失败 - {str(e)}")
        else:
            print(f"✗ {gesture}: 文件不存在 - {filename}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\n总计: {len(combined_df)} 样本, {len(class_counts)} 个类别")
        return combined_df, all_labels, class_counts
    else:
        print("\n错误: 没有成功加载任何数据!")
        return None, None, None

# 加载数据
df, labels, class_counts = load_all_data(CONFIG['data_path'], GESTURE_CLASSES)

# 显示类别分布
if df is not None:
    plt.figure(figsize=(15, 5))
    plt.bar(class_counts.keys(), class_counts.values())
    plt.xlabel('Gesture Class')
    plt.ylabel('Sample Count')
    plt.title('Data Distribution by Gesture Class')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 第三步：数据预处理与特征工程

In [ ]:
class GestureDataPreprocessor:
    """
    手势数据预处理器
    
    功能:
    1. 传感器数据归一化
    2. 卡尔曼滤波平滑
    3. 滑动窗口序列生成
    4. 数据增强
    """
    
    def __init__(self, sequence_length=100, sampling_rate=50):
        self.sequence_length = sequence_length
        self.sampling_rate = sampling_rate
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        
    def normalize_flex(self, flex_data, min_val=0, max_val=4095):
        """
        弯曲传感器归一化到[0,1]
        
        公式: x_norm = (x - min) / (max - min)
        """
        return (flex_data - min_val) / (max_val - min_val)
    
    def normalize_imu(self, accel, gyro):
        """
        IMU数据归一化
        
        加速度: 归一化到g (除以9.81)
        陀螺仪: 归一化到[-1,1] (除以2000 deg/s)
        """
        accel_norm = accel / 9.81
        gyro_norm = gyro / 2000.0
        return accel_norm, gyro_norm
    
    def apply_kalman_filter(self, data, process_noise=0.01, measurement_noise=0.1):
        """
        应用一维卡尔曼滤波
        
        状态方程: x_k = x_{k-1} + w_k  (w ~ N(0, Q))
        观测方程: z_k = x_k + v_k      (v ~ N(0, R))
        
        卡尔曼增益: K = P_pred / (P_pred + R)
        状态更新: x = x_pred + K * (z - x_pred)
        协方差更新: P = (1 - K) * P_pred
        """
        n_iter = len(data)
        xhat = np.zeros(n_iter)
        P = np.zeros(n_iter)
        
        Q = process_noise  # 过程噪声协方差
        R = measurement_noise  # 测量噪声协方差
        
        # 初始化
        xhat[0] = data[0]
        P[0] = 1.0
        
        for k in range(1, n_iter):
            # 预测步骤
            x_pred = xhat[k-1]
            P_pred = P[k-1] + Q
            
            # 更新步骤
            K = P_pred / (P_pred + R)
            xhat[k] = x_pred + K * (data[k] - x_pred)
            P[k] = (1 - K) * P_pred
        
        return xhat
    
    def preprocess_features(self, df):
        """
        预处理特征数据
        """
        feature_cols = [
            'flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5',
            'ACCx', 'ACCy', 'ACCz',
            'GYRx', 'GYRy', 'GYRz'
        ]
        
        features = df[feature_cols].values
        
        # 归一化弯曲传感器 (前5列)
        flex_max = features[:, :5].max()
        features[:, :5] = features[:, :5] / flex_max if flex_max > 0 else features[:, :5]
        
        # 归一化加速度 (列5-7)
        features[:, 5:8] = features[:, 5:8] / 9.81
        
        # 归一化陀螺仪 (列8-10)
        gyro_max = np.abs(features[:, 8:11]).max()
        features[:, 8:11] = features[:, 8:11] / gyro_max if gyro_max > 0 else features[:, 8:11]
        
        return features
    
    def create_sequences(self, data, labels, stride=10):
        """
        创建时间序列样本
        
        参数:
            data: 特征数据 (N, F)
            labels: 标签 (N,)
            stride: 滑动窗口步长
        
        返回:
            sequences: (M, T, F)
            sequence_labels: (M,)
        """
        sequences = []
        sequence_labels = []
        
        # 按标签分组处理
        unique_labels = np.unique(labels)
        
        for label in unique_labels:
            # 获取该标签的数据
            mask = labels == label
            label_data = data[mask]
            
            # 生成序列
            for i in range(0, len(label_data) - self.sequence_length, stride):
                seq = label_data[i:i + self.sequence_length]
                sequences.append(seq)
                sequence_labels.append(label)
        
        return np.array(sequences), np.array(sequence_labels)
    
    def augment_sequence(self, sequence, noise_factor=0.05):
        """
        数据增强: 添加高斯噪声
        
        x' = x + ε, ε ~ N(0, σ²)
        """
        noise = np.random.normal(0, noise_factor, sequence.shape)
        return sequence + noise
    
    def time_warp(self, sequence, sigma=0.2, num_knots=4):
        """
        时间扭曲增强
        """
        from scipy.interpolate import CubicSpline
        
        orig_steps = np.arange(sequence.shape[0])
        random_warps = np.random.normal(loc=1.0, scale=sigma, size=(num_knots+2))
        warp_steps = np.linspace(0, sequence.shape[0]-1, num=num_knots+2)
        
        cs = CubicSpline(warp_steps, random_warps)
        warped_steps = cs(orig_steps)
        warped_steps = np.clip(warped_steps, 0, sequence.shape[0]-1)
        
        warped_sequence = np.zeros_like(sequence)
        for i in range(sequence.shape[1]):
            warped_sequence[:, i] = np.interp(
                orig_steps, warped_steps, sequence[:, i]
            )
        
        return warped_sequence

# 初始化预处理器
preprocessor = GestureDataPreprocessor(
    sequence_length=CONFIG['sequence_length'],
    sampling_rate=CONFIG['sampling_rate']
)

print("预处理器初始化完成")
print(f"序列长度: {preprocessor.sequence_length}")
print(f"采样率: {preprocessor.sampling_rate}Hz")

In [ ]:
# 执行预处理
if df is not None:
    print("开始数据预处理...")
    
    # 1. 预处理特征
    features = preprocessor.preprocess_features(df)
    labels = df['Alphabet'].values
    
    print(f"特征矩阵形状: {features.shape}")
    print(f"标签数量: {len(labels)}")
    
    # 2. 创建时间序列
    print("\n创建时间序列...")
    X, y = preprocessor.create_sequences(
        features, 
        labels, 
        stride=CONFIG['stride']
    )
    
    print(f"序列数据形状: {X.shape}")
    print(f"序列标签形状: {y.shape}")
    
    # 3. 编码标签
    y_encoded = preprocessor.label_encoder.fit_transform(y)
    num_classes = len(preprocessor.label_encoder.classes_)
    
    print(f"\n类别编码:")
    for i, cls in enumerate(preprocessor.label_encoder.classes_):
        print(f"  {cls} -> {i}")
    
    # 4. 划分训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded,
        test_size=CONFIG['test_size'],
        random_state=CONFIG['random_state'],
        stratify=y_encoded
    )
    
    print(f"\n数据集划分:")
    print(f"  训练集: {X_train.shape[0]} 样本")
    print(f"  测试集: {X_test.shape[0]} 样本")
    print(f"  特征维度: {X_train.shape[1:]}")
    print(f"  类别数: {num_classes}")

## 第四步：Conv1D + Bi-LSTM模型设计

In [ ]:
def create_gesture_model(sequence_length, num_features, num_classes, 
                         dropout_rate=0.3, lstm_units=128):
    """
    创建Conv1D + Bi-LSTM手势识别模型
    
    架构:
    1. Conv1D层: 提取局部时序特征
       - Conv1D(32, kernel=3) + BN + MaxPool + Dropout
       - Conv1D(64, kernel=3) + BN + MaxPool + Dropout
       - Conv1D(128, kernel=3) + BN
    
    2. Bi-LSTM层: 捕捉长期依赖
       - Bi-LSTM(128, return_sequences=True) + Dropout
       - Bi-LSTM(64, return_sequences=False) + Dropout
    
    3. 分类层:
       - Dense(128) + ReLU + Dropout
       - Dense(num_classes) + Softmax
    
    参数:
        sequence_length: 时间序列长度 (T)
        num_features: 特征维度 (F)
        num_classes: 类别数 (C)
        dropout_rate: Dropout比率
        lstm_units: LSTM单元数
    """
    
    model = tf.keras.Sequential([
        # 输入层
        tf.keras.layers.Input(shape=(sequence_length, num_features)),
        
        # ===== Conv1D特征提取层 =====
        # 第1层 Conv1D
        tf.keras.layers.Conv1D(
            filters=32, 
            kernel_size=3, 
            activation='relu',
            padding='same',
            name='conv1d_1'
        ),
        tf.keras.layers.BatchNormalization(name='bn_1'),
        tf.keras.layers.MaxPooling1D(pool_size=2, name='maxpool_1'),
        tf.keras.layers.Dropout(dropout_rate, name='dropout_1'),
        
        # 第2层 Conv1D
        tf.keras.layers.Conv1D(
            filters=64, 
            kernel_size=3, 
            activation='relu',
            padding='same',
            name='conv1d_2'
        ),
        tf.keras.layers.BatchNormalization(name='bn_2'),
        tf.keras.layers.MaxPooling1D(pool_size=2, name='maxpool_2'),
        tf.keras.layers.Dropout(dropout_rate, name='dropout_2'),
        
        # 第3层 Conv1D
        tf.keras.layers.Conv1D(
            filters=128, 
            kernel_size=3, 
            activation='relu',
            padding='same',
            name='conv1d_3'
        ),
        tf.keras.layers.BatchNormalization(name='bn_3'),
        
        # ===== Bi-LSTM层 =====
        # 第1层 Bi-LSTM (返回序列)
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(
                lstm_units, 
                return_sequences=True,
                dropout=dropout_rate,
                name='lstm_1'
            ),
            name='bilstm_1'
        ),
        tf.keras.layers.Dropout(dropout_rate, name='dropout_3'),
        
        # 第2层 Bi-LSTM (返回最后一个输出)
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(
                lstm_units // 2,  # 减半单元数
                return_sequences=False,
                dropout=dropout_rate,
                name='lstm_2'
            ),
            name='bilstm_2'
        ),
        tf.keras.layers.Dropout(dropout_rate, name='dropout_4'),
        
        # ===== 分类层 =====
        tf.keras.layers.Dense(128, activation='relu', name='dense_1'),
        tf.keras.layers.Dropout(dropout_rate, name='dropout_5'),
        tf.keras.layers.Dense(
            num_classes, 
            activation='softmax', 
            name='output'
        )
    ])
    
    return model

# 创建模型
if 'X_train' in locals():
    model = create_gesture_model(
        sequence_length=CONFIG['sequence_length'],
        num_features=CONFIG['num_features'],
        num_classes=num_classes,
        dropout_rate=0.3,
        lstm_units=128
    )
    
    # 显示模型摘要
    print("模型架构:")
    print("="*80)
    model.summary()
    
    # 计算参数量
    total_params = model.count_params()
    print(f"\n总参数量: {total_params:,}")
    print(f"模型大小估计: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")

In [ ]:
# 可视化模型架构
if 'model' in locals():
    tf.keras.utils.plot_model(
        model, 
        to_file='model_architecture.png',
        show_shapes=True,
        show_layer_names=True,
        dpi=100
    )
    
    # 显示图片
    from IPython.display import Image, display
    display(Image('model_architecture.png'))

## 第五步：模型训练

In [ ]:
# 配置训练参数
TRAINING_CONFIG = {
    'batch_size': 32,
    'epochs': 100,
    'learning_rate': 0.001,
    'patience': 10,  # Early stopping patience
    'reduce_lr_patience': 5,
    'min_lr': 1e-6
}

# 编译模型
if 'model' in locals():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=TRAINING_CONFIG['learning_rate']
        ),
        loss='sparse_categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')
        ]
    )
    
    print("模型编译完成")
    print(f"优化器: Adam (lr={TRAINING_CONFIG['learning_rate']})")
    print(f"损失函数: sparse_categorical_crossentropy")
    print(f"评估指标: accuracy, top3_accuracy")

In [ ]:
# 设置回调函数
if 'model' in locals():
    callbacks = [
        # Early Stopping
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=TRAINING_CONFIG['patience'],
            restore_best_weights=True,
            verbose=1
        ),
        
        # Reduce Learning Rate on Plateau
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=TRAINING_CONFIG['reduce_lr_patience'],
            min_lr=TRAINING_CONFIG['min_lr'],
            verbose=1
        ),
        
        # Model Checkpoint
        tf.keras.callbacks.ModelCheckpoint(
            'best_model.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        
        # TensorBoard (可选)
        # tf.keras.callbacks.TensorBoard(log_dir='./logs')
    ]
    
    print("回调函数设置完成:")
    for cb in callbacks:
        print(f"  - {cb.__class__.__name__}")

In [ ]:
# 开始训练
if 'model' in locals() and 'X_train' in locals():
    print("\n开始训练...")
    print("="*80)
    
    history = model.fit(
        X_train, y_train,
        batch_size=TRAINING_CONFIG['batch_size'],
        epochs=TRAINING_CONFIG['epochs'],
        validation_split=0.2,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\n训练完成!")
    print(f"最终训练精度: {history.history['accuracy'][-1]:.4f}")
    print(f"最终验证精度: {history.history['val_accuracy'][-1]:.4f}")

In [ ]:
# 绘制训练曲线
if 'history' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 准确率
    axes[0, 0].plot(history.history['accuracy'], label='Train')
    axes[0, 0].plot(history.history['val_accuracy'], label='Validation')
    axes[0, 0].set_title('Model Accuracy')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # 损失
    axes[0, 1].plot(history.history['loss'], label='Train')
    axes[0, 1].plot(history.history['val_loss'], label='Validation')
    axes[0, 1].set_title('Model Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Top-3 准确率
    if 'top3_accuracy' in history.history:
        axes[1, 0].plot(history.history['top3_accuracy'], label='Train')
        axes[1, 0].plot(history.history['val_top3_accuracy'], label='Validation')
        axes[1, 0].set_title('Top-3 Accuracy')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Accuracy')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
    
    # 学习率
    if 'lr' in history.history:
        axes[1, 1].plot(history.history['lr'])
        axes[1, 1].set_title('Learning Rate')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].set_yscale('log')
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()

## 第六步：模型评估

In [ ]:
# 在测试集上评估
if 'model' in locals() and 'X_test' in locals():
    print("评估测试集性能...")
    print("="*80)
    
    # 评估
    test_loss, test_acc, test_top3 = model.evaluate(
        X_test, y_test, verbose=0
    )
    
    print(f"测试集损失: {test_loss:.4f}")
    print(f"测试集准确率: {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"测试集Top-3准确率: {test_top3:.4f} ({test_top3*100:.2f}%)")
    
    # 预测
    y_pred = model.predict(X_test, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # 分类报告
    print("\n分类报告:")
    print("-"*80)
    print(classification_report(
        y_test, 
        y_pred_classes,
        target_names=preprocessor.label_encoder.classes_
    ))

In [ ]:
# 绘制混淆矩阵
if 'y_pred_classes' in locals():
    cm = confusion_matrix(y_test, y_pred_classes)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=preprocessor.label_encoder.classes_,
        yticklabels=preprocessor.label_encoder.classes_
    )
    plt.title('Confusion Matrix', fontsize=16)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()
    
    # 归一化混淆矩阵
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm_norm, 
        annot=True, 
        fmt='.2f', 
        cmap='Blues',
        xticklabels=preprocessor.label_encoder.classes_,
        yticklabels=preprocessor.label_encoder.classes_
    )
    plt.title('Normalized Confusion Matrix', fontsize=16)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('confusion_matrix_norm.png', dpi=150)
    plt.show()

## 第七步：TensorFlow Lite转换

In [ ]:
def representative_dataset_gen():
    """
    生成代表性数据集用于INT8量化
    
    TFLite量化需要一个代表性数据集来估计
    激活值的动态范围
    """
    for i in range(min(100, len(X_train))):
        # 选择训练集中的样本
        sample = X_train[i:i+1].astype(np.float32)
        yield [sample]

def convert_to_tflite(model, output_dir='./tflite_model', model_name='gesture_model'):
    """
    转换模型为TensorFlow Lite格式
    
    生成三种格式:
    1. 标准TFLite (float32)
    2. 动态范围量化 (混合精度)
    3. INT8全量化 (推荐用于ESP32)
    
    参数:
        model: Keras模型
        output_dir: 输出目录
        model_name: 模型名称前缀
    """
    
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    results = {}
    
    # 1. 标准TFLite (float32)
    print("1. 转换标准TFLite模型 (float32)...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    
    float_path = os.path.join(output_dir, f'{model_name}_float32.tflite')
    with open(float_path, 'wb') as f:
        f.write(tflite_model)
    
    results['float32'] = {
        'path': float_path,
        'size_kb': len(tflite_model) / 1024
    }
    print(f"   大小: {results['float32']['size_kb']:.1f} KB")
    
    # 2. 动态范围量化
    print("\n2. 转换动态范围量化模型...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model_quant = converter.convert()
    
    dynamic_path = os.path.join(output_dir, f'{model_name}_dynamic.tflite')
    with open(dynamic_path, 'wb') as f:
        f.write(tflite_model_quant)
    
    results['dynamic'] = {
        'path': dynamic_path,
        'size_kb': len(tflite_model_quant) / 1024
    }
    print(f"   大小: {results['dynamic']['size_kb']:.1f} KB")
    
    # 3. INT8全量化 (推荐)
    print("\n3. 转换INT8全量化模型 (推荐用于ESP32)...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    
    tflite_model_int8 = converter.convert()
    
    int8_path = os.path.join(output_dir, f'{model_name}_int8.tflite')
    with open(int8_path, 'wb') as f:
        f.write(tflite_model_int8)
    
    results['int8'] = {
        'path': int8_path,
        'size_kb': len(tflite_model_int8) / 1024
    }
    print(f"   大小: {results['int8']['size_kb']:.1f} KB")
    
    print(f"\n模型已保存到: {output_dir}/")
    return results

# 执行转换
if 'model' in locals():
    print("开始TensorFlow Lite转换...")
    print("="*80)
    
    tflite_results = convert_to_tflite(
        model, 
        output_dir='./tflite_model',
        model_name='gesture_model'
    )
    
    # 比较模型大小
    print("\n模型大小对比:")
    print("-"*80)
    for model_type, info in tflite_results.items():
        print(f"{model_type:10s}: {info['size_kb']:6.1f} KB")

In [ ]:
# 生成C++头文件
def generate_cpp_header(tflite_model_path, output_path='./gesture_model.h', model_name='gesture_model'):
    """
    将TFLite模型转换为C++头文件
    
    适用于ESP32的TFLite Micro部署
    """
    
    # 读取模型文件
    with open(tflite_model_path, 'rb') as f:
        model_data = f.read()
    
    # 生成C++代码
    cpp_code = f'''
#pragma once
// Auto-generated TFLite model header
// Model: {model_name}
// Size: {len(model_data)} bytes
// Generated: {pd.Timestamp.now()}

#include <cstdint>

// Model data array
const unsigned char {model_name}_tflite[] = {{
'''
    
    # 将二进制数据转换为十六进制字符串
    hex_lines = []
    for i in range(0, len(model_data), 12):
        chunk = model_data[i:i+12]
        hex_values = [f'0x{b:02x}' for b in chunk]
        hex_lines.append('    ' + ', '.join(hex_values) + ',')
    
    cpp_code += '\n'.join(hex_lines)
    
    cpp_code += f'''
}};

// Model size in bytes
const int {model_name}_tflite_len = {len(model_data)};

// Model metadata
const int MODEL_SEQUENCE_LENGTH = {CONFIG['sequence_length']};
const int MODEL_NUM_FEATURES = {CONFIG['num_features']};
const int MODEL_NUM_CLASSES = {num_classes if 'num_classes' in locals() else 0};
'''
    
    # 保存文件
    with open(output_path, 'w') as f:
        f.write(cpp_code)
    
    print(f"C++头文件已生成: {output_path}")
    print(f"  模型大小: {len(model_data)} bytes ({len(model_data)/1024:.1f} KB)")
    
    return output_path

# 生成INT8模型的头文件
if 'tflite_results' in locals():
    header_path = generate_cpp_header(
        tflite_results['int8']['path'],
        output_path='./gesture_model.h',
        model_name='gesture_model'
    )

## 第八步：ESP32部署代码

In [ ]:
# 生成ESP32部署代码
esp32_main_cpp = '''
/**
 * Smart Glove ASL Gesture Recognition
 * ESP32-S3 Deployment Code
 * 
 * Model: Conv1D + Bi-LSTM
 * Input: 11 features x 100 timesteps
 * Output: Gesture class probabilities
 */

#include <Arduino.h>
#include <TensorFlowLite_ESP32.h>
#include <Adafruit_MPU6050.h>
#include <Adafruit_Sensor.h>
#include <Wire.h>

// Include the model
#include "gesture_model.h"

// ============== Configuration ==============
#define SEQUENCE_LENGTH  100
#define NUM_FEATURES     11
#define NUM_CLASSES      ''' + str(num_classes if 'num_classes' in locals() else 22) + '''
#define SAMPLING_RATE_HZ 50
#define SAMPLING_PERIOD_MS (1000 / SAMPLING_RATE_HZ)

// Sensor pins
const int FLEX_PINS[5] = {36, 39, 34, 35, 32};

// ============== TensorFlow Lite Globals ==============
namespace {
  const tflite::Model* model = nullptr;
  tflite::MicroInterpreter* interpreter = nullptr;
  TfLiteTensor* input = nullptr;
  TfLiteTensor* output = nullptr;
  
  // Tensor Arena - 根据模型调整大小
  constexpr int kTensorArenaSize = 50 * 1024;  // 50KB
  alignas(16) uint8_t tensor_arena[kTensorArenaSize];
}

// ============== Sensor Globals ==============
Adafruit_MPU6050 mpu;

// Sliding window buffer
float sensor_buffer[SEQUENCE_LENGTH][NUM_FEATURES];
int buffer_index = 0;
bool buffer_full = false;

// ============== Kalman Filter ==============
class KalmanFilter {
private:
  float Q, R, P, X, K;
  
public:
  KalmanFilter(float process_noise = 0.01, float measurement_noise = 0.1) {
    Q = process_noise;
    R = measurement_noise;
    P = 1.0;
    X = 0.0;
    K = 0.0;
  }
  
  float update(float measurement) {
    float X_pred = X;
    float P_pred = P + Q;
    K = P_pred / (P_pred + R);
    X = X_pred + K * (measurement - X_pred);
    P = (1 - K) * P_pred;
    return X;
  }
};

KalmanFilter kf_flex[5];
KalmanFilter kf_accel[3];
KalmanFilter kf_gyro[3];

// ============== Function Declarations ==============
void setupModel();
void setupSensors();
void readSensors(float* features);
int predictGesture();
void printGesture(int class_idx, float confidence);

// ============== Setup ==============
void setup() {
  Serial.begin(115200);
  while (!Serial) delay(10);
  
  Serial.println("\n========================================");
  Serial.println("Smart Glove ASL Recognition");
  Serial.println("Conv1D + Bi-LSTM on ESP32-S3");
  Serial.println("========================================\n");
  
  // Initialize sensors
  setupSensors();
  
  // Initialize TensorFlow Lite model
  setupModel();
  
  Serial.println("\nSystem Ready!\n");
}

// ============== Main Loop ==============
void loop() {
  static unsigned long last_sample_time = 0;
  unsigned long current_time = millis();
  
  // Sample at 50Hz
  if (current_time - last_sample_time >= SAMPLING_PERIOD_MS) {
    last_sample_time = current_time;
    
    // Read sensor features
    float features[NUM_FEATURES];
    readSensors(features);
    
    // Add to sliding window buffer
    for (int i = 0; i < NUM_FEATURES; i++) {
      sensor_buffer[buffer_index][i] = features[i];
    }
    
    buffer_index = (buffer_index + 1) % SEQUENCE_LENGTH;
    if (buffer_index == 0) buffer_full = true;
    
    // Run inference when buffer is full
    if (buffer_full) {
      int gesture_class = predictGesture();
      
      if (gesture_class >= 0) {
        // Get confidence
        float* predictions = output->data.f;
        float confidence = predictions[gesture_class];
        
        printGesture(gesture_class, confidence);
      }
    }
  }
}

// ============== Function Implementations ==============

void setupModel() {
  Serial.println("Initializing TensorFlow Lite model...");
  
  // Load model
  model = tflite::GetModel(gesture_model_tflite);
  
  if (model->version() != TFLITE_SCHEMA_VERSION) {
    Serial.printf("Model schema version %d not supported. Expected %d\n",
                  model->version(), TFLITE_SCHEMA_VERSION);
    return;
  }
  
  // Create resolver
  static tflite::MicroMutableOpResolver<12> resolver;
  resolver.AddConv2D();
  resolver.AddMaxPool2D();
  resolver.AddLstm();
  resolver.AddFullyConnected();
  resolver.AddSoftmax();
  resolver.AddQuantize();
  resolver.AddDequantize();
  resolver.AddReshape();
  resolver.AddConcatenation();
  resolver.AddBatchNorm();
  resolver.AddRelu();
  resolver.AddAdd();
  
  // Create interpreter
  static tflite::MicroInterpreter static_interpreter(
    model, resolver, tensor_arena, kTensorArenaSize
  );
  interpreter = &static_interpreter;
  
  // Allocate tensors
  TfLiteStatus allocate_status = interpreter->AllocateTensors();
  if (allocate_status != kTfLiteOk) {
    Serial.println("AllocateTensors() failed!");
    return;
  }
  
  input = interpreter->input(0);
  output = interpreter->output(0);
  
  Serial.println("Model initialized successfully!");
  Serial.printf("  Input shape: %dx%d\n", SEQUENCE_LENGTH, NUM_FEATURES);
  Serial.printf("  Output classes: %d\n", NUM_CLASSES);
  Serial.printf("  Tensor arena: %d bytes\n", kTensorArenaSize);
}

void setupSensors() {
  Serial.println("Initializing sensors...");
  
  // Initialize I2C
  Wire.begin();
  
  // Initialize MPU6050
  if (!mpu.begin(0x68, &Wire)) {
    Serial.println("Failed to find MPU6050!");
    while (1) delay(10);
  }
  
  mpu.setAccelerometerRange(MPU6050_RANGE_8_G);
  mpu.setGyroRange(MPU6050_RANGE_500_DEG);
  mpu.setFilterBandwidth(MPU6050_BAND_21_HZ);
  
  // Initialize flex sensor pins
  for (int i = 0; i < 5; i++) {
    pinMode(FLEX_PINS[i], INPUT);
  }
  
  Serial.println("Sensors initialized!");
}

void readSensors(float* features) {
  // Read flex sensors (5 channels)
  for (int i = 0; i < 5; i++) {
    int raw = analogRead(FLEX_PINS[i]);
    float normalized = raw / 4095.0;  // Normalize to [0, 1]
    features[i] = kf_flex[i].update(normalized);
  }
  
  // Read IMU
  sensors_event_t a, g, temp;
  mpu.getEvent(&a, &g, &temp);
  
  // Accelerometer (3 channels) - normalize to g
  features[5] = kf_accel[0].update(a.acceleration.x / 9.81);
  features[6] = kf_accel[1].update(a.acceleration.y / 9.81);
  features[7] = kf_accel[2].update(a.acceleration.z / 9.81);
  
  // Gyroscope (3 channels) - normalize
  features[8] = kf_gyro[0].update(g.gyro.x / 2000.0);
  features[9] = kf_gyro[1].update(g.gyro.y / 2000.0);
  features[10] = kf_gyro[2].update(g.gyro.z / 2000.0);
}

int predictGesture() {
  // Copy buffer to input tensor
  // Note: TFLite Micro expects flattened input
  for (int t = 0; t < SEQUENCE_LENGTH; t++) {
    int idx = (buffer_index + t) % SEQUENCE_LENGTH;
    for (int f = 0; f < NUM_FEATURES; f++) {
      input->data.f[t * NUM_FEATURES + f] = sensor_buffer[idx][f];
    }
  }
  
  // Run inference
  TfLiteStatus invoke_status = interpreter->Invoke();
  if (invoke_status != kTfLiteOk) {
    Serial.println("Invoke failed!");
    return -1;
  }
  
  // Find max probability class
  float* predictions = output->data.f;
  int predicted_class = 0;
  float max_prob = predictions[0];
  
  for (int i = 1; i < NUM_CLASSES; i++) {
    if (predictions[i] > max_prob) {
      max_prob = predictions[i];
      predicted_class = i;
    }
  }
  
  return predicted_class;
}

void printGesture(int class_idx, float confidence) {
  // Map class index to gesture name
  // Update this mapping based on your training labels
  const char* gesture_names[] = {
    ''' + ', '.join(['"' + cls + '"' for cls in (GESTURE_CLASSES if 'GESTURE_CLASSES' in locals() else ['gesture_' + str(i) for i in range(22)])]) + '''
  };
  
  Serial.printf("Gesture: %s (Confidence: %.2f%%)\n",
                gesture_names[class_idx],
                confidence * 100);
}
'''

# 保存ESP32代码
with open('./esp32_main.cpp', 'w') as f:
    f.write(esp32_main_cpp)

print("ESP32部署代码已生成: ./esp32_main.cpp")
print(f"  代码行数: {len(esp32_main_cpp.splitlines())}")

In [ ]:
# 生成PlatformIO配置文件
platformio_ini = '''
; PlatformIO Project Configuration File
; Smart Glove ASL Recognition with Conv1D + Bi-LSTM

[platformio]
default_envs = esp32-s3

[env:esp32-s3]
platform = espressif32
board = esp32-s3-devkitc-1
framework = arduino

; Monitor settings
monitor_speed = 115200
monitor_filters = esp32_exception_decoder

; Upload settings
upload_speed = 921600

; Library dependencies
lib_deps = 
    adafruit/Adafruit MPU6050 @ ^2.2.0
    adafruit/Adafruit Sensor @ ^1.1.6
    adafruit/Adafruit BusIO @ ^1.14.0
    https://github.com/tanakamasayuki/TensorFlowLite_ESP32.git

; Build flags
build_flags = 
    -DCORE_DEBUG_LEVEL=3
    -DCONFIG_ARDUHAL_LOG_COLORS=1
    -DBOARD_HAS_PSRAM
    -mfix-esp32-psram-cache-issue
    -O2

; Partition table for large models
board_build.partitions = default_16MB.csv

; Memory type (for ESP32-S3 with PSRAM)
board_build.arduino.memory_type = qio_qspi
'''

with open('./platformio.ini', 'w') as f:
    f.write(platformio_ini)

print("PlatformIO配置文件已生成: ./platformio.ini")

## 第九步：总结与导出

In [ ]:
# 生成项目总结报告
import json

summary = {
    "project": "Smart Glove ASL Recognition",
    "model_architecture": "Conv1D + Bi-LSTM",
    "configuration": CONFIG if 'CONFIG' in locals() else {},
    "training_config": TRAINING_CONFIG if 'TRAINING_CONFIG' in locals() else {},
    "performance": {
        "test_accuracy": float(test_acc) if 'test_acc' in locals() else None,
        "test_top3_accuracy": float(test_top3) if 'test_top3' in locals() else None,
    },
    "model_size": {
        "keras_params": int(model.count_params()) if 'model' in locals() else 0,
        "tflite_sizes": {k: f"{v['size_kb']:.1f} KB" for k, v in tflite_results.items()} if 'tflite_results' in locals() else {}
    },
    "deployment": {
        "target": "ESP32-S3",
        "framework": "Arduino",
        "tensor_arena": "50KB",
        "files_generated": [
            "gesture_model.h",
            "esp32_main.cpp",
            "platformio.ini",
            "gesture_model_int8.tflite"
        ]
    }
}

# 保存JSON报告
with open('./project_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("项目总结报告已生成: ./project_summary.json")
print("\n报告内容:")
print(json.dumps(summary, indent=2))

## 生成的文件清单

运行本Notebook后将生成以下文件:

1. **Model Files**:
   - `best_model.h5` - 最佳Keras模型
   - `gesture_model_float32.tflite` - 浮点TFLite模型
   - `gesture_model_dynamic.tflite` - 动态量化模型
   - `gesture_model_int8.tflite` - INT8量化模型(推荐)

2. **Deployment Files**:
   - `gesture_model.h` - C++头文件(包含模型数据)
   - `esp32_main.cpp` - ESP32主程序
   - `platformio.ini` - PlatformIO配置文件

3. **Documentation**:
   - `model_architecture.png` - 模型架构图
   - `training_history.png` - 训练曲线
   - `confusion_matrix.png` - 混淆矩阵
   - `project_summary.json` - 项目总结

## 下一步

1. **部署到ESP32**:
   - 将 `gesture_model.h` 和 `esp32_main.cpp` 复制到PlatformIO项目的 `src/` 目录
   - 将 `platformio.ini` 复制到项目根目录
   - 编译并上传到ESP32-S3

2. **测试与验证**:
   - 验证传感器数据读取
   - 测试推理延迟
   - 调整置信度阈值

3. **优化**:
   - 收集更多训练数据
   - 尝试不同的模型架构
   - 优化Tensor Arena大小